<a href="https://colab.research.google.com/github/SandeepKonduruFeb12/aiml/blob/master/gold/GoldAssignment3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task
Train a small Causal Language Model (LLM) from scratch using the `transformers` library on the 'wikitext-2-raw-v1' dataset, demonstrating the setup, training, and text generation capabilities of the model.

## Install Libraries
Install all necessary libraries including `transformers`, `datasets`, `accelerate`, and `torch` in the Colab environment.


In [1]:
pip install transformers datasets accelerate torch

## Load and Prepare Dataset

### Subtask:
Load a suitable dataset like 'wikitext-2-raw-v1' using the `datasets` library. Load a compatible tokenizer (e.g., 'gpt2' tokenizer). Tokenize the dataset and prepare it for causal language modeling, ensuring texts are grouped and processed into sequences of a fixed length suitable for training.


**Reasoning**:
I need to load the specified dataset, initialize a tokenizer, tokenize the dataset, and then group the tokenized texts into fixed-length sequences for causal language modeling. All these steps can be efficiently performed within a single code block using the `datasets` and `transformers` libraries.



In [6]:
from datasets import load_dataset
from transformers import AutoTokenizer

# 2. Load the 'wikitext-2-raw-v1' dataset
raw_datasets = load_dataset('wikitext', 'wikitext-2-raw-v1')
print("Dataset loaded successfully:")
print(raw_datasets)

# 3. Import the AutoTokenizer class and instantiate a tokenizer
tokenizer = AutoTokenizer.from_pretrained('gpt2')

# Add a pad token if it doesn't exist (useful for some models like GPT2 when batching)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': tokenizer.eos_token})
    print(f"Added pad token: {tokenizer.pad_token}")

# 4. Define a preprocessing function to tokenize the text data
def tokenize_function(examples):
    # Handle cases where text might be empty or None
    texts = [text if text is not None else '' for text in examples["text"]]
    return tokenizer(texts)

# 5. Apply the preprocessing function to the entire dataset
tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched=True,
    num_proc=4, # Use multiple processes for faster tokenization
    remove_columns=raw_datasets["train"].column_names, # Remove original 'text' column
)
print("Tokenized datasets:")
print(tokenized_datasets)

# 6. Define a block size for grouping texts
block_size = 128

# 7. Define a function to group texts into blocks of the specified block_size
def group_texts(examples):
    # Concatenate all texts from a batch
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop operation.
    # For now, we just drop the last partial block.
    total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

# 8. Apply this grouping function to the tokenized dataset
lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    batch_size=1000,
    num_proc=4,
    # Remove columns that are no longer needed after grouping
    remove_columns=["token_type_ids", "attention_mask"] if "token_type_ids" in tokenized_datasets["train"].column_names else ["attention_mask"]
)
print("Language modeling datasets prepared:")
print(lm_datasets)
print(f"Sample features of a training example: {lm_datasets['train'][0].keys()}")

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

...

Tokenized datasets:
DatasetDict({
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 3760
    })
})


Map (num_proc=4):   0%|          | 0/4358 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/36718 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/3760 [00:00<?, ? examples/s]

Language modeling datasets prepared:
DatasetDict({
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2209
    })
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 18666
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1931
    })
})
Sample features of a training example: dict_keys(['input_ids', 'attention_mask', 'labels'])


## Configure and Initialize LLM

### Subtask:
Define a small `AutoConfig` (e.g., `GPT2Config`) to create a reasonably sized transformer model from scratch. Initialize an `AutoModelForCausalLM` or `GPT2LMHeadModel` with this configuration.


**Reasoning**:
The subtask requires defining a small GPT2Config, initializing a Causal Language Model with it, and then printing the number of parameters. I will use the `transformers` library for this.



In [7]:
from transformers import TrainingArguments, DataCollatorForLanguageModeling

# 2. Define output_dir for saving model checkpoints and logs
output_dir = "llm_finetuning_results"

# 3. Create a TrainingArguments instance
training_args = TrainingArguments(
    output_dir=output_dir,
    overwrite_output_dir=True,
    num_train_epochs=3,                 # A small number of epochs for demonstration
    per_device_train_batch_size=8,      # Small batch size
    per_device_eval_batch_size=8,
    learning_rate=2e-5,                 # Standard learning rate
    weight_decay=0.01,
    logging_steps=50,                   # Log every 50 steps
    eval_strategy="epoch",              # Evaluate at the end of each epoch (renamed from evaluation_strategy)
    save_strategy="epoch",              # Save model at the end of each epoch (renamed from save_strategy)
    load_best_model_at_end=True,        # Load the best model after training
    metric_for_best_model="eval_loss",  # Metric to use for early stopping and best model loading
    greater_is_better=False,            # Lower eval_loss is better
    report_to="none"                    # Do not report to any online service
)

print("Training arguments defined:")
print(training_args)

# 4. Initialize a DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print("Data collator initialized for causal language modeling.")

Training arguments defined:
TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
e

## Initialize and Train Model

### Subtask:
Initialize a Trainer with the configured model, training arguments, tokenized training dataset, and a data collator. Then, start the training process.


**Reasoning**:
I need to initialize a `Trainer` object with the previously defined model, training arguments, and datasets, then initiate the training process and evaluate the model, as per the subtask instructions.



In [8]:
from transformers import Trainer, AutoConfig, AutoModelForCausalLM
import math

# Define configuration for a small GPT-2 model based on previous subtask requirements
config = AutoConfig.from_pretrained(
    "gpt2",
    vocab_size=len(tokenizer),
    n_positions=block_size,
    n_ctx=block_size,
    n_embd=128,  # Small embedding size
    n_layer=2,   # Small number of layers
    n_head=2     # Small number of attention heads
)
print("Model configuration defined:")
print(config)

# Initialize a model from the defined configuration
model = AutoModelForCausalLM.from_config(config)
print("Model initialized.")
print(f"Number of model parameters: {model.num_parameters()}")

# 2. Initialize a Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["validation"],
    data_collator=data_collator,
)

print("Trainer initialized.")

# 3. Start the training process
train_result = trainer.train()
print("Training complete.")

# 4. Evaluate the model and calculate perplexity
metrics = trainer.evaluate()
loss = metrics["eval_loss"]
perplexity = math.exp(loss)

print(f"Evaluation Loss: {loss:.2f}")
print(f"Perplexity: {perplexity:.2f}")

Model configuration defined:
GPT2Config {
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 128,
  "n_embd": 128,
  "n_head": 2,
  "n_inner": null,
  "n_layer": 2,
  "n_positions": 128,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "transformers_version": "4.57.3",
  "use_cache": true,
  "vocab_size": 50257
}

Model initialized.
Number of model parameters: 6846080
Trainer initialized.


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,7.438200,7.406615
2,7.117500,7.104465
3,7.051600,7.029530


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Training complete.


Evaluation Loss: 7.03
Perplexity: 1129.50


## Generate Text

### Subtask:
Utilize the trained model to generate coherent text, showcasing its learned language patterns. This demonstrates the practical application of the fine-tuning process.

In [9]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import os # Import os module for path checking

# Ensure tokenizer is defined. If not, load it.
try:
    _ = tokenizer  # Check if tokenizer exists
except NameError:
    print("Tokenizer not found. Loading tokenizer from 'gpt2'.")
    tokenizer = AutoTokenizer.from_pretrained('gpt2')
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({'pad_token': tokenizer.eos_token})

# Ensure model is defined. If not, attempt to load the trained model.
try:
    _ = model # Check if model exists
except NameError:
    print("Model not found in current session. Attempting to load trained model from disk.")
    output_dir = "llm_finetuning_results" # This was already corrected

    # Explicitly check if the output directory exists
    if not os.path.exists(output_dir):
        print(f"Error: The directory '{output_dir}' does not exist locally.\nThis usually means the model training cell (ID 67325ee0) has not been executed or failed to save the model.")
        print("Please ensure you run the model training cell (ID 67325ee0) first to train and save the model.")
        raise RuntimeError("Trained model directory not found. Please run the training cell (ID 67325ee0).")

    try:
        model = AutoModelForCausalLM.from_pretrained(output_dir)
        print(f"Trained model loaded from {output_dir}.")
    except Exception as e:
        print(f"Could not load trained model from {output_dir}. Error: {e}")
        print("Please ensure the 'Initialize and Train Model' cell (ID 67325ee0) has been executed to train and save the model.")
        # Exit or raise error if model cannot be loaded and cannot proceed
        raise RuntimeError("Failed to load trained model. Please run previous cells.")

# Create a text generation pipeline using the fine-tuned model and tokenizer
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

# Define a prompt to start text generation
prompt = "The quick brown fox"

# Generate text, explicitly setting max_new_tokens to a value within the model's context length
print(f"\nGenerating text with prompt: '{prompt}'")

generated_text = generator(prompt, max_new_tokens=50, num_return_sequences=1)

for i, result in enumerate(generated_text):
    print(f"Generated text {i+1}: {result['generated_text']}")

# Try another prompt
prompt_2 = "In a world of magic and dragons"
print(f"\nGenerating text with prompt: '{prompt_2}'")
generated_text_2 = generator(prompt_2, max_new_tokens=70, num_return_sequences=1)

for i, result in enumerate(generated_text_2):
    print(f"Generated text {i+1}: {result['generated_text']}")

Device set to use cuda:0



Generating text with prompt: 'The quick brown fox'
Generated text 1: The quick brown fox. 
's
 = 
 = = = 
 The " to the ", on the the@ for the
 The of the = = = = = = = = = =, was the = = = The the " 

Generating text with prompt: 'In a world of magic and dragons'
Generated text 1: In a world of magic and dragons,@, and the first. 
 in the was. 
 = =
 = 
 to the by the he = = = = = = ( the the and the by the
 = = = 
 The
 = = = =  in the was the. 
 = = = 
 = = = = The

